# Image formation, measured

Companion notebook for the CondadosAI Fundamentals unit **How a camera forms an image**
(F1 / unit 1.1). Every code block printed in the four lesson posts has a cell here, so
nothing on the page is a snippet you cannot run.

Sources used, none of them redistributed by this repository:

| Lesson | Source | Licence |
|---|---|---|
| 1, 3 | OpenCV `samples/data/left*.jpg` | Apache-2.0 |
| 2 | capture settings printed on DPDD's `figures/data_example.png` | MIT |
| 4 | `jakarto3d/py-OCamCalib` `test_images/fish_1` | **GPL-2.0**, fetched, never vendored |


In [ ]:
# Pinned so a Colab base-image change cannot silently move the numbers.
!pip install -q 'opencv-python>=5.0.0,<6.0.0' 'numpy>=1.26'


In [ ]:
import urllib.request
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

# The posts' cells call show(image, title); in a notebook that is a matplotlib figure.
def show(img, title=''):
    plt.figure(figsize=(10, 4))
    plt.imshow(img[..., ::-1] if img.ndim == 3 else img, cmap='gray')
    plt.title(title)
    plt.axis('off')
    plt.show()

print('OpenCV', cv2.__version__, '| NumPy', np.__version__)


## Fetch the photographs

The calibration set skips `left10`: a loop over `range(1, 15)` downloads a 404 page
instead of a photograph, and `findChessboardCorners` then fails on a file that looks fine.


In [ ]:
OPENCV_RAW = 'https://raw.githubusercontent.com/opencv/opencv/4.x/samples/data'
LEFT_IDS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14]

calib_dir = Path('data/opencv-calib'); calib_dir.mkdir(parents=True, exist_ok=True)
paths = []
for i in LEFT_IDS:
    name = f'left{i:02d}.jpg'
    dst = calib_dir / name
    if not dst.exists():
        urllib.request.urlretrieve(f'{OPENCV_RAW}/{name}', dst)
    assert dst.stat().st_size > 5000, f'{name} came back too small to be a photograph'
    paths.append(dst)
print(len(paths), 'calibration images')

# the posts' cells read the first photo under this name (lossless, same pixels)
cv2.imwrite('left01.webp', cv2.imread(str(paths[0])), [cv2.IMWRITE_WEBP_QUALITY, 101])


## Calibrate

The board has 9x6 interior corners. The physical square size is not published with these
images, so this runs in board units: `fx`, `fy`, the principal point and every distortion
coefficient are unchanged by that, and only the translation vectors carry a unit.


In [ ]:
BOARD = (9, 6)
CRITERIA = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 1e-3)

objp = np.zeros((BOARD[0] * BOARD[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:BOARD[0], 0:BOARD[1]].T.reshape(-1, 2)

obj_points, img_points = [], []
for path in paths:
    gray = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2GRAY)
    found, corners = cv2.findChessboardCorners(gray, BOARD, None)
    if not found:
        print('board not found in', path.name); continue
    obj_points.append(objp)
    img_points.append(cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), CRITERIA))

size = (gray.shape[1], gray.shape[0])
rms, K, dist, rvecs, tvecs = cv2.calibrateCamera(obj_points, img_points, size, None, None)
print(f'fx={K[0,0]:.2f} fy={K[1,1]:.2f} cx={K[0,2]:.2f} cy={K[1,2]:.2f} RMS={rms:.4f} px')
print('k1, k2, p1, p2, k3 =', np.round(dist.ravel(), 5))


## Lesson 1 — the pinhole equations

The block printed in [the pinhole lesson](https://condados.ai/blog/pinhole-and-perspective-projection).
`(X, Y, Z)` is a point in the camera's own frame.


In [ ]:
fx, fy = K[0, 0], K[1, 1]
cx, cy = K[0, 2], K[1, 2]

rot, _ = cv2.Rodrigues(rvecs[0])
camera_frame = (rot @ objp.T + tvecs[0]).T

# The lesson works the corner the pinhole equations miss by the most, so pick that
# one rather than an arbitrary index.
pinhole = np.column_stack([cx + fx * camera_frame[:, 0] / camera_frame[:, 2],
                           cy + fy * camera_frame[:, 1] / camera_frame[:, 2]])
measured = img_points[0].reshape(-1, 2)
error = np.linalg.norm(pinhole - measured, axis=1)
worst = int(np.argmax(error))

X, Y, Z = camera_frame[worst]
u = cx + fx * X / Z
v = cy + fy * Y / Z
print(f'camera frame ({X:.4f}, {Y:.4f}, {Z:.4f}) -> pinhole ({u:.2f}, {v:.2f})')
print(f'the detector found it at ({measured[worst][0]:.2f}, {measured[worst][1]:.2f}), '
      f'off by {error[worst]:.2f} px')


In [ ]:
# The whole view: how the residual is distributed across the frame.
radius = np.linalg.norm(measured - np.array([cx, cy]), axis=1)
order = np.argsort(radius); third = len(order) // 3
print(f'mean {error.mean():.2f} px, max {error.max():.2f} px')
print(f'innermost third {error[order[:third]].mean():.2f} px, '
      f'outermost third {error[order[-third:]].mean():.2f} px')


### Lesson 1: how close is this to OpenCV?

In [ ]:
import numpy as np

# the calibrated camera, from the unit's 13 photos
fx, fy = 536.0735, 536.0164
cx, cy = 342.3705, 235.5369

# one board corner in the camera's own frame
X, Y, Z = 4.68658, -4.06741, 13.83412

u = cx + fx * X / Z
v = cy + fy * Y / Z
print(f"pinhole: ({u:.2f}, {v:.2f})")
print("measured: (513.77, 86.53)")

In [ ]:
import cv2

K = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]])
dist = np.array([-0.26509, -0.04674, 0.00183,
                 -0.00031, 0.25231])
pt = np.array([[X, Y, Z]])
zero = np.zeros(3)

# no distortion: the same two equations
p, _ = cv2.projectPoints(pt, zero, zero, K, None)
print("cv2, pinhole only:", p.ravel().round(2))
# with the lens's measured bend
q, _ = cv2.projectPoints(pt, zero, zero, K, dist)
print("cv2, with distortion:", q.ravel().round(2))
for name, a in (("pinhole", p), ("with bend", q)):
    e = np.hypot(*(a.ravel() - [513.77, 86.53]))
    print(f"{name:9} misses the corner by {e:.2f} px")

### Lesson 1: exercises

In [ ]:
def project(X, Y, Z):
    return (0.0, 0.0)  # pixel (u, v), pinhole only

print(project(X, Y, Z))

In [ ]:
# checks your answer
print("correct" if (np.allclose(project(X, Y, Z), (523.98, 77.94), atol=0.01)) else "not yet")

In [ ]:
# Exercises 2 and 3: the numbers under "What you should see"
for scale in (1, 2):
    u = cx + fx * X / (scale * Z)
    v = cy + fy * Y / (scale * Z)
    print(f"depth x{scale}: ({u:.2f}, {v:.2f}), "
          f"{np.hypot(u - cx, v - cy):.1f} px from the centre")
print("frame middle to optical centre:",
      round(cx - 320, 1), "px across,",
      round(240 - cy, 1), "px up")

## Lesson 2 — depth of field

The function printed in [the lens lesson](https://condados.ai/blog/lenses-focal-length-depth-of-field).
No pixels are involved: the inputs are the capture settings the DPDD figure publishes.


In [ ]:
def dof_limits(f_mm, n, subject_m, coc_mm=0.02476):
    """Near and far limits of acceptable focus, in metres."""
    f = f_mm / 1000
    a = f * f / (n * coc_mm / 1000)          # hyperfocal minus f
    u = subject_m - f
    far = float('inf') if u >= a else subject_m * a / (a - u)
    return subject_m * a / (a + u), far

for n in (4, 22):
    near, far = dof_limits(93, n, 1.5246)
    print(f'f/{n}: {near:.3f} - {far:.3f} m  (depth {far - near:.3f} m)')


### Lesson 2: how close is this to OpenCV?

In [ ]:
def dof_limits(f_mm, n, subject_m, coc_mm=0.02476):
    """Near and far limits of acceptable focus, in metres."""
    f = f_mm / 1000
    a = f * f / (n * coc_mm / 1000)   # hyperfocal minus f
    u = subject_m - f
    far = (float("inf") if u >= a
           else subject_m * a / (a - u))
    return subject_m * a / (a + u), far

# DPDD prints 1.50-1.55 m at f/4; recover what it does not print
s = 2 * 1.50 * 1.55 / (1.50 + 1.55)   # subject distance
for n in (4, 22):
    near, far = dof_limits(93, n, s)
    print(f"f/{n}: {near:.3f} to {far:.3f} m, "
          f"depth {far - near:.3f} m")

In [ ]:
# against the published figure and the aperture ratio
near4, far4 = dof_limits(93, 4, s)
near22, far22 = dof_limits(93, 22, s)
print("f/4 published 1.50 to 1.55, computed",
      round(near4, 2), "to", round(far4, 2))
print("depth ratio:",
      round((far22 - near22) / (far4 - near4), 2),
      " aperture ratio:", 22 / 4)

### Lesson 2: exercises

In [ ]:
def hyperfocal(f_mm, n, coc_mm=0.02476):
    return 0.0  # metres

print(hyperfocal(93, 4))

In [ ]:
# checks your answer
print("correct" if (abs(hyperfocal(93, 4) - 87.42) < 0.05) else "not yet")

In [ ]:
# Exercises 2 and 3: the numbers under "What you should see"
d4 = far4 - near4
for n in (8, 11):
    a, b = dof_limits(93, n, s)
    print(f"f/{n}: depth {b - a:.3f} m, "
          f"{(b - a) / d4:.2f} times f/4")
a, b = dof_limits(93, 4, 3.0)
print(f"subject at 3 m, f/4: {a:.3f} to {b:.3f} m, "
      f"depth {b - a:.3f} m")

## Lesson 3 — undistorting

The block printed in [the distortion lesson](https://condados.ai/blog/lens-distortion-vignetting-aberration),
plus the measurement that shows what it costs.


In [ ]:
img = cv2.imread(str(paths[0]))
undistorted = cv2.undistort(img, K, dist)          # one frame

h, w = img.shape[:2]                                # for video, build the map once
new_k, _ = cv2.getOptimalNewCameraMatrix(K, dist, (w, h), 0)
map1, map2 = cv2.initUndistortRectifyMap(K, dist, None, new_k, (w, h), cv2.CV_16SC2)
undistorted = cv2.remap(img, map1, map2, cv2.INTER_LINEAR)
print('undistorted', undistorted.shape)


In [ ]:
# What it costs: field of view, not pixels.
def hfov(f, width): return float(2 * np.degrees(np.arctan((width / 2) / f)))

for label, nk in (('same f', K.copy()),
                  ('alpha=0', cv2.getOptimalNewCameraMatrix(K, dist, (w, h), 0)[0]),
                  ('alpha=1', cv2.getOptimalNewCameraMatrix(K, dist, (w, h), 1)[0])):
    mx, my = cv2.initUndistortRectifyMap(K, dist, None, nk, (w, h), cv2.CV_32FC1)
    empty = ((mx < 0) | (mx > w - 1) | (my < 0) | (my > h - 1)).mean() * 100
    print(f'{label:8s} hfov {hfov(nk[0,0], w):.2f} deg, {empty:.2f}% of the output has no source')


### Lesson 3: how close is this to OpenCV?

In [ ]:
import numpy as np

fx, fy = 536.0735, 536.0164
cx, cy = 342.3705, 235.5369
k1, k2, k3 = -0.26509, -0.04674, 0.25231
p1, p2 = 0.00183, -0.00031

# the top-left pixel, centred and scaled by f
x, y = (0 - cx) / fx, (0 - cy) / fy
r2 = x * x + y * y
radial = 1 + k1 * r2 + k2 * r2**2 + k3 * r2**3
xd = x * radial + 2 * p1 * x * y + p2 * (r2 + 2 * x * x)
yd = y * radial + p1 * (r2 + 2 * y * y) + 2 * p2 * x * y
ud, vd = cx + fx * xd, cy + fy * yd
print(f"radial factor {radial:.5f}")
print(f"lands at ({ud:.2f}, {vd:.2f}), "
      f"{np.hypot(ud, vd):.2f} px from (0, 0)")

In [ ]:
import cv2

K = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]])
dist = np.array([k1, k2, p1, p2, k3])

# OpenCV's forward model on the same ray
p, _ = cv2.projectPoints(np.array([[x, y, 1.0]]),
                         np.zeros(3), np.zeros(3), K, dist)
print("cv2.projectPoints:", p.ravel().round(2))

# and back: undistortPoints inverts it iteratively
back = cv2.undistortPoints(p.reshape(1, 1, 2), K, dist, P=K)
print("cv2.undistortPoints:", back.ravel().round(2))

img = cv2.imread("left01.webp")
show(np.hstack([img, cv2.undistort(img, K, dist)]),
     "left01: as taken, and straightened")

### Lesson 3: exercises

In [ ]:
def shift_px(u, v):
    return 0.0  # how far the lens moves pixel (u, v)

print(shift_px(0, 0))

In [ ]:
# checks your answer
print("correct" if (abs(shift_px(0, 0) - 51.22) < 0.05 and shift_px(cx, cy) < 1e-9) else "not yet")

In [ ]:
# Exercises 2 and 3: the numbers under "What you should see"
def moved(u, v, radial_only=False):
    x, y = (u - cx) / fx, (v - cy) / fy
    r2 = x * x + y * y
    f = 1 + k1 * r2 + k2 * r2**2 + k3 * r2**3
    xd, yd = x * f, y * f
    if not radial_only:
        xd += 2 * p1 * x * y + p2 * (r2 + 2 * x * x)
        yd += p1 * (r2 + 2 * y * y) + 2 * p2 * x * y
    return np.hypot(cx + fx * xd - u, cy + fy * yd - v)

for name, (u, v) in (("left edge, middle row", (0, 240)),
                     ("halfway to the corner", (171, 118)),
                     ("top-left corner", (0, 0))):
    print(f"{name:22} {moved(u, v):6.2f} px")
print("corner, radial terms only:",
      round(moved(0, 0, True), 2), "px")

## Lesson 4 — the fisheye fit

The block printed in [the fisheye lesson](https://condados.ai/blog/wide-angle-and-fisheye-models).

**Licence note:** these 15 photographs are GPL-2.0. They are downloaded here and not
redistributed; if you republish anything derived from them, that licence applies to you.


In [ ]:
FISHEYE_RAW = 'https://raw.githubusercontent.com/jakarto3d/py-OCamCalib/main/test_images/fish_1'
fish_dir = Path('data/fisheye-fish1'); fish_dir.mkdir(parents=True, exist_ok=True)
fish_paths = []
for i in range(1, 16):
    dst = fish_dir / f'Fisheye1_{i}.jpg'
    if not dst.exists():
        urllib.request.urlretrieve(f'{FISHEYE_RAW}/{dst.name}', dst)
    fish_paths.append(dst)

# 6x8 interior corners, not the 5x7 the source repository's own file states.
FISH_BOARD = (6, 8)
fobjp = np.zeros((FISH_BOARD[0] * FISH_BOARD[1], 3), np.float64)
fobjp[:, :2] = np.mgrid[0:FISH_BOARD[0], 0:FISH_BOARD[1]].T.reshape(-1, 2)

fobj, fimg = [], []
for path in fish_paths:
    gray = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2GRAY)
    found, corners = cv2.findChessboardCorners(gray, FISH_BOARD, None)
    if not found:
        continue
    refined = cv2.cornerSubPix(gray, corners, (5, 5), (-1, -1), CRITERIA).astype(np.float64)
    fobj.append(fobjp.reshape(1, -1, 3))
    fimg.append(refined.reshape(1, -1, 2))
fsize = (gray.shape[1], gray.shape[0])
print(f'board found in {len(fobj)}/{len(fish_paths)} views')


In [ ]:
h, w = fsize[1], fsize[0]
# Without a starting guess the solver converges to a focal length it cannot
# recover from. f = max(w, h) / pi is the equidistant estimate for a 180 degree lens.
k = np.array([[max(w, h) / np.pi, 0, w / 2], [0, max(w, h) / np.pi, h / 2], [0, 0, 1]])
flags = cv2.CALIB_USE_INTRINSIC_GUESS | cv2.CALIB_RECOMPUTE_EXTRINSIC | cv2.CALIB_FIX_SKEW
rms_f, k, d, rvecs_f, tvecs_f = cv2.fisheye.calibrate(
    fobj, fimg, fsize, k, np.zeros((4, 1)),
    [np.zeros((1, 1, 3)) for _ in fobj], [np.zeros((1, 1, 3)) for _ in fobj], flags,
    (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 200, 1e-8))
print(f'Kannala-Brandt: f={k[0,0]:.2f} px, RMS {rms_f:.3f} px, coefficients {np.round(d.ravel(), 6)}')


In [ ]:
# The same corners, fitted with the model lessons 1-3 taught.
rms_p, Kp, distp, _, _ = cv2.calibrateCamera(
    [o.reshape(-1, 3).astype(np.float32) for o in fobj],
    [i.reshape(-1, 1, 2).astype(np.float32) for i in fimg], fsize, None, None)
print(f'pinhole + Brown-Conrady: f={Kp[0,0]:.2f} px, RMS {rms_p:.3f} px')
print(f'ratio {rms_p / rms_f:.1f}x')


### Lesson 4: how close is this to OpenCV?

In [ ]:
import numpy as np

# pinhole + Brown-Conrady, fitted to the 15 fisheye photos
f_pin = 372.2176
k1, k2, k3 = -0.1904280, 0.02420791, -0.001105085
# Kannala-Brandt, fitted to the same corners
f_fish = 336.8584
kb = [-0.0026441, -0.00029514, -0.0031234, 0.00034040]

def r_pinhole(deg):
    r = np.tan(np.radians(deg))
    r2 = r * r
    return f_pin * r * (1 + k1*r2 + k2*r2**2 + k3*r2**3)

def r_fisheye(deg):
    t = np.radians(deg)
    td = t * (1 + kb[0]*t**2 + kb[1]*t**4
              + kb[2]*t**6 + kb[3]*t**8)
    return f_fish * td

for deg in (15, 45, 60, 70, 80):
    print(f"{deg:2} deg  pinhole {r_pinhole(deg):10.1f}"
          f"   fisheye {r_fisheye(deg):6.1f} px")

In [ ]:
import cv2

zero = np.zeros(3)
for deg in (45, 80):
    t = np.radians(deg)
    ray = np.array([[[np.sin(t), 0, np.cos(t)]]])
    K = np.array([[f_fish, 0, 0], [0, f_fish, 0],
                  [0, 0, 1.0]])
    q, _ = cv2.fisheye.projectPoints(
        ray, zero, zero, K, np.array(kb))
    Kp = np.array([[f_pin, 0, 0], [0, f_pin, 0],
                   [0, 0, 1.0]])
    d = np.array([k1, k2, 0, 0, k3])
    p, _ = cv2.projectPoints(ray[0], zero, zero, Kp, d)
    print(f"{deg} deg  cv2.projectPoints "
          f"{p.ravel()[0]:10.1f}   "
          f"cv2.fisheye {q.ravel()[0]:6.1f} px")

### Lesson 4: exercises

In [ ]:
def equidistant(deg, f=336.8584):
    return 0.0  # radius on the sensor, px

print(equidistant(80))

In [ ]:
# checks your answer
print("correct" if (abs(equidistant(80) - 470.34) < 0.05) else "not yet")

In [ ]:
# Exercises 2 and 3: the numbers under "What you should see"
for deg in (45, 80):
    eq = f_fish * np.radians(deg)
    print(f"{deg} deg: fitted {r_fisheye(deg):.1f} px, "
          f"equidistant {eq:.1f} px, "
          f"{(r_fisheye(deg) / eq - 1) * 100:+.2f}%")
for deg in (60, 80):
    print(f"{deg} deg: tan stretches "
          f"{1 / np.cos(np.radians(deg))**2:.1f} times "
          f"the centre")

## Your own camera

Print a chessboard, photograph it from a dozen angles, upload them, and re-run the
calibration cell with `paths` pointing at your files. Everything downstream is the same
code the articles publish.


In [ ]:
# from google.colab import files
# uploaded = files.upload()
# paths = [Path(name) for name in uploaded]
